In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. データ準備
data = fetch_covtype()
X = data.data
y = data.target - 1  # ラベルを0-6に変更（PyTorch用）

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Tensor変換
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

# DataLoader作成
batch_size = 256
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size)

# 2. モデル定義
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.net(x)

# 3. 学習関数（ミニバッチ学習）
def train_model(train_loader, val_loader, input_dim, hidden_dim, output_dim, lr=0.001, epochs=10):
    model = MLP(input_dim, hidden_dim, output_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()

        # 検証精度の計算
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb).argmax(dim=1)
                all_preds.extend(pred.cpu().numpy())
                all_labels.extend(yb.cpu().numpy())
        acc = accuracy_score(all_labels, all_preds)
        if acc > best_acc:
            best_acc = acc
            best_state = model.state_dict()
    
    return best_acc, best_state

# 4. グリッドサーチ
hidden_dims = [64, 128]
learning_rates = [0.001, 0.01]
results = []

for h in hidden_dims:
    for lr in learning_rates:
        acc, state = train_model(train_loader, val_loader, X_train.shape[1], h, 7, lr)
        results.append((h, lr, acc, state))
        print(f"Hidden={h}, LR={lr} => Val Acc={acc:.4f}")

# 5. 最良モデルでテスト
best_h, best_lr, _, best_state = max(results, key=lambda x: x[2])
model = MLP(X_train.shape[1], best_h, 7).to(device)
model.load_state_dict(best_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(dim=1)
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
print(f"\nBest Hyperparameters: hidden={best_h}, lr={best_lr}")
print(f"Test Accuracy: {test_acc:.4f}")

# Classification report
from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds))

Using device: cuda:1
Hidden=64, LR=0.001 => Val Acc=0.7982
Hidden=64, LR=0.01 => Val Acc=0.8175
Hidden=128, LR=0.001 => Val Acc=0.8175
Hidden=128, LR=0.01 => Val Acc=0.8382

Best Hyperparameters: hidden=128, lr=0.01
Test Accuracy: 0.8313
              precision    recall  f1-score   support

           0       0.85      0.78      0.82     42368
           1       0.83      0.89      0.86     56661
           2       0.85      0.78      0.81      7151
           3       0.72      0.70      0.71       549
           4       0.72      0.44      0.55      1899
           5       0.60      0.77      0.67      3473
           6       0.89      0.81      0.85      4102

    accuracy                           0.83    116203
   macro avg       0.78      0.74      0.75    116203
weighted avg       0.83      0.83      0.83    116203

